In [1]:
import pandas as pd
import numpy as np


In [ ]:
books = pd.read_csv("data_cleaned.csv")

In [ ]:
books.head()

In [ ]:
books['categories'].head()

In [ ]:
books['categories'].value_counts().reset_index()

In [ ]:
books['categories'].value_counts().reset_index().query("count >50")

In [ ]:
books[books['categories'] == "Juvenile Fiction"]

In [ ]:
books[books['categories'] == "Juvenile Nonfiction"]

In [ ]:
category_mapping = {'Fiction' : "Fiction",
 'Juvenile Fiction': "Children's Fiction",
 'Biography & Autobiography': "Nonfiction",
 'History': "Nonfiction",
 'Literary Criticism': "Nonfiction",
 'Philosophy': "Nonfiction",
 'Religion': "Nonfiction",
 'Comics & Graphic Novels': "Fiction",
 'Drama': "Fiction",
 'Juvenile Nonfiction': "Children's Nonfiction",
 'Science': "Nonfiction",
 'Poetry': "Fiction"}

books["simple_categories"] = books["categories"].map(category_mapping)

In [ ]:
books["simple_categories"].isna().sum()

In [ ]:
books[~(books["simple_categories"].isna())]


In [ ]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]

pipe = pipeline("zero-shot-classification",
                model="facebook/bart-large-mnli"
               )

In [ ]:
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[0]


In [ ]:
pipe(sequence, fiction_categories)

In [ ]:
# Max index will give the index of the highest probability
max_index = np.argmax(pipe(sequence, fiction_categories)["scores"])
max_label = pipe(sequence, fiction_categories)["labels"][max_index]
max_label

In [ ]:
def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label

## Let look at how good this model is at generating the predictions


In [ ]:
from tqdm import tqdm

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Fiction"]

In [ ]:
for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Nonfiction"]

In [ ]:
predictions_df = pd.DataFrame({"actual_cat": actual_cats, "predicted_cat": predicted_cats})

In [ ]:
predictions_df.head()

In [ ]:
predictions_df["correct_pred"] = (
    np.where(predictions_df["actual_cat"] == predictions_df["predicted_cat"], 1, 0)
)

In [ ]:
predictions_df.head()

In [ ]:
predictions_df["correct_pred"].sum() / len(predictions_df)


In [ ]:
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [ ]:
missing_cats.head(3)

In [ ]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    isbns += [missing_cats["isbn13"][i]]

In [ ]:
missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_cats": predicted_cats})
missing_predicted_df.head()

In [ ]:
books.columns

In [ ]:
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left")
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_cats"], books["simple_categories"])
books = books.drop(columns = ["predicted_cats"])

In [ ]:
books.head()

In [ ]:
books[books["categories"].str.lower().isin([
    "romance",
    "science fiction",
    "scifi",
    "fantasy",
    "horror",
    "mystery",
    "thriller",
    "comedy",
    "crime",
    "historical"
])]

In [ ]:
books.to_csv("books_with_categories.csv", index=False)